# Offline embedding — InsuranceRAG

Runs the bi-encoder on a Colab GPU because a 0.6B model over ~2,900 chunks is hours on CPU.
Nothing else about the pipeline moves: chunks in, vectors out, Postgres stays on your laptop.

**In** `data/chunks/*.jsonl` (from `scripts/ingest.py`), zipped and uploaded.
**Out** `data/embeddings/{doc_id}.npz`, holding `ids` and `vectors`, downloaded back.

Then locally: `python scripts/index.py --embeddings data/embeddings`.

In [23]:
!pip install -q sentence-transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [24]:
rm -f chunks*.zip

In [26]:
!ls -la *.zip

-rw-r--r-- 1 root root 7062037 Sep  4 22:06 embeddings.zip


## 1. Upload the chunks


In [27]:
from google.colab import files

files.upload()  # pick chunks.zip
!rm -rf chunks embeddings && mkdir -p chunks embeddings
!unzip -q -j chunks.zip -d chunks
!ls -la chunks

Saving chunks.zip to chunks.zip
total 2952
drwxr-xr-x 2 root root    4096 Sep  4 22:11 .
drwxr-xr-x 1 root root    4096 Sep  4 22:11 ..
-rw-rw-rw- 1 root root  119753 Sep  4 21:42 auto-insurance-rro-664.jsonl
-rw-rw-rw- 1 root root   43489 Sep  4 21:42 court-proceedings-o-reg-461-96.jsonl
-rw-rw-rw- 1 root root   49854 Sep  4 21:42 fault-rules-rro-668.jsonl
-rw-rw-rw- 1 root root    4268 Sep  4 21:42 fsra-attendant-care-rate.jsonl
-rw-rw-rw- 1 root root    6670 Sep  4 21:42 fsra-attendant-care-rate-revised.jsonl
-rw-rw-rw- 1 root root   24040 Sep  4 21:42 fsra-indexation-amounts.jsonl
-rw-rw-rw- 1 root root   47474 Sep  4 21:42 fsra-minor-injury-guideline.jsonl
-rw-rw-rw- 1 root root   15002 Sep  4 21:42 fsra-transportation-expense.jsonl
-rw-rw-rw- 1 root root 1985062 Sep  4 21:42 insurance-act-part-vi.jsonl
-rw-rw-rw- 1 root root  235324 Sep  4 21:42 oap-1.jsonl
-rw-rw-rw- 1 root root  460007 Sep  4 21:42 sabs-o-reg-34-10.jsonl


## 2. Load the encoder

fp16 on a T4 halves memory and roughly doubles throughput; the vectors are cast back to
float32 before saving so pgvector receives the same precision it would locally.

In [28]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"  # must equal settings.bi_encoder_model
BATCH_SIZE = 32

model = SentenceTransformer(
    MODEL_ID, device="cuda", model_kwargs={"torch_dtype": "float16"}
)

model.max_seq_length = 1024  # chunks are capped at 800 reference tokens
print(model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

1024


/tmp/ipykernel_1602/3552573187.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(model.get_sentence_embedding_dimension())


## 3. Embed, one document at a time

Per-document files, written as each finishes — a Colab disconnect then costs one document,
not the whole run. Re-running skips whatever is already on disk.

In [29]:
import json
from pathlib import Path

import numpy as np

CHUNKS = Path("chunks")
OUT = Path("embeddings")

for path in sorted(CHUNKS.glob("*.jsonl")):
    target = OUT / f"{path.stem}.npz"
    if target.exists():
        print(f"{path.stem:<40} cached")
        continue

    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    vectors = model.encode(
        [r["text"] for r in records],
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    np.savez(target, ids=np.array([r["chunk_id"] for r in records]), vectors=vectors)
    print(f"{path.stem:<40} {vectors.shape}")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

auto-insurance-rro-664                   (129, 1024)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

court-proceedings-o-reg-461-96           (43, 1024)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

insurance-act-part-vi                    (1980, 1024)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

oap-1                                    (221, 1024)


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

sabs-o-reg-34-10                         (395, 1024)


## 4. Download

Unzip into `data/embeddings/` locally, then run `python scripts/index.py --embeddings data/embeddings`.

In [30]:
!zip -q -r embeddings.zip embeddings
files.download("embeddings.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>